# Advent of Code - Day 9

Find the Day 9 problem [here](https://adventofcode.com/2025/day/9").

## Pre-requisites: specify import(s), set constants, and load data.

In [ ]:
import re
from itertools import combinations
from scipy.optimize import milp, LinearConstraint, Bounds
import numpy as np

In [ ]:
# Data constants.
PUZZLE_FILENAME = "data/day10.txt"
TEXT_ENCODING = "utf8"

In [ ]:
# Solution constants.
FAIL = -1

In [ ]:
def load_data() -> list[str]:
    """Loads data from the file and puts it in a line-by-line format.

    Example data:
    [.##.] (3) (1,3) (2) (2,3) (0,2) (0,1) {3,5,4,7}
    [...#.] (0,2,3,4) (2,3) (0,4) (0,1,2) (1,2,3,4) {7,5,12,7,2}
    [.###.#] (0,1,2,3,4) (0,3,4) (0,1,2,4,5) (1,2) {10,11,11,5,10,5}
    
    Returns: The content of the text file line-by-line.
    """
    with open(PUZZLE_FILENAME, 'r', encoding=TEXT_ENCODING) as file:
        full_file = file.read()
        lines = full_file.strip().split('\n')

    return lines

def parse_line(line: list[str]) -> tuple[list[str], list[str]]:
    """Parse a single line of data into key components."""
    indicator_match = re.search(r'\[([.#]+)\]', line)
    indicator = indicator_match.group(1)
    
    buttons = re.findall(r'\(([0-9,]+)\)', line)

    joltage_match = re.search(r'\{([0-9,]+)\}', line)
    joltage = [int(x) for x in joltage_match.group(1).split(',')]
    
    return indicator, buttons, joltage

# Part One

### Notes
* All lights are initially OFF.

In [ ]:
def solve_single_machine(indicator: list[str], buttons: list[str]) -> int:
    """Find minimum number of button presses to achieve target indicator pattern."""
    n_lights = len(indicator)
    
    target = tuple(1 if c == '#' else 0 for c in indicator)
    
    # Convert button specifications to toggle vectors.
    button_vectors = []
    for button in buttons:
        vec = [0] * n_lights
        indices = [int(x) for x in button.split(',')]
        for idx in indices:
            vec[idx] = 1
        button_vectors.append(tuple(vec))
    
    n_buttons = len(button_vectors)
    
    # Try subsets of increasing size.
    for num_presses in range(n_buttons + 1):
        for combo in combinations(range(n_buttons), num_presses):
            # XOR all selected button vectors together
            result = [0] * n_lights
            for btn_idx in combo:
                for i in range(n_lights):
                    result[i] ^= button_vectors[btn_idx][i]
            
            if tuple(result) == target:
                return num_presses

    return -1

def calculate_fewest_button_presses() -> int:
    """Calculate the fewest button presses required for correct config."""
    
    lines = load_data()
    total = 0
    
    for line in lines:
        line = line.strip()
        if line:
            indicator, buttons, _ = parse_line(line)
            presses = solve_single_machine(indicator, buttons)
            if presses >= 0:
                total += presses
            else:
                print(f"Warning: No solution for line: {line}")
    
    return total

### Get Solution to Part One

In [ ]:
print(f"Fewest number of button presses: {calculate_fewest_button_presses()}")

# Part Two

In [ ]:
def solve_single_machine_part_2(buttons, joltage) -> int:
    """Find minimum button presses to achieve target joltage levels."""
    n_counters = len(joltage)
    n_buttons = len(buttons)
    
    # Build the constraint matrix A where A[i][j] = 1 if button j affects counter i.
    A = np.zeros((n_counters, n_buttons))
    for j, button in enumerate(buttons):
        indices = [int(x) for x in button.split(',')]
        for idx in indices:
            A[idx][j] = 1
    
    c = np.ones(n_buttons)
    constraints = LinearConstraint(A, joltage, joltage)
    bounds = Bounds(lb=0, ub=np.inf)
    integrality = np.ones(n_buttons)
    result = milp(c, constraints=constraints, bounds=bounds, integrality=integrality)
    
    if result.success:
        return int(round(result.fun))
    else:
        return FAIL

def calculate_fewest_button_presses_for_joltage_level_counters() -> int:
    """Solve for all machines and return total minimum presses for joltage level counters."""
    lines = load_data()
    total = 0
    
    for line in lines:
        line = line.strip()
        if line:
            _, buttons, joltage = parse_line(line)
            presses = solve_single_machine_part_2(buttons, joltage)
            if presses >= 0:
                total += presses
            else:
                print(f"Warning: Error on line: {line}")
    
    return total

### Get Solution to Part Two

In [ ]:
print(f"Fewest button presses to configure joltage level counters: {calculate_fewest_button_presses_for_joltage_level_counters()}")